Exclude the Rows with with "Import"

In [10]:
# Match "Import" regardless of capitalization or extra spaces
import pandas as pd

df = pd.read_csv("commodity_trade_statistics_data.csv")

flow = df["flow"].astype("string").str.strip().str.lower()
df_no_imports = df.loc[~flow.eq("import").fillna(False)]

display(df_no_imports)
print(f"Rows remaining: {len(df_no_imports):,}")

/var/folders/lk/rjp1f34x7nsdq5kqy58g4nl80000gn/T/ipykernel_54989/2120942355.py:4: DtypeWarning: Columns (0: comm_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("commodity_trade_statistics_data.csv")


,country_or_area,year,comm_code,commodity,flow,trade_usd,weight_kg,quantity_name,quantity,category
0,Afghanistan,2016,10410,"Sheep, live",Export,6088,2339.0,Number of items,51.0,01_live_animals
1,Afghanistan,2016,10420,"Goats, live",Export,3958,984.0,Number of items,53.0,01_live_animals
6,Albania,2016,10511,"Fowls, live domestic < 185 grams",Export,87581,5320.0,Number of items,115180.0,01_live_animals
10,Albania,2016,10600,"Animals, live, except farm animals",Export,584025,59521.0,Number of items,59521.0,01_live_animals
18,Albania,2015,10600,"Animals, live, except farm animals",Export,636184,80774.0,Number of items,80694.0,01_live_animals
...,...,...,...,...,...,...,...,...,...,...
8225864,Zimbabwe,2002,TOTAL,ALL COMMODITIES,Re-Export,115886192,NaN,No Quantity,NaN,all_commodities
8225866,Zimbabwe,2001,TOTAL,ALL COMMODITIES,Export,1206807424,NaN,No Quantity,NaN,all_commodities
8225867,Zimbabwe,2001,TOTAL,ALL COMMODITIES,Re-Export,15943977,NaN,No Quantity,NaN,all_commodities
8225868,Zimbabwe,2000,TOTAL,ALL COMMODITIES,Export,1924962432,NaN,No Quantity,NaN,all_commodities


Rows remaining: 3,377,347


Count number of rows per year per country

In [9]:
country_year_counts = (
    df_no_imports.groupby(["country_or_area", "year"], dropna=False)
    .size()
    .reset_index(name="row_count")
)

display(country_year_counts)

,country_or_area,year,row_count
0,Afghanistan,2008,34
1,Afghanistan,2009,31
2,Afghanistan,2010,33
3,Afghanistan,2011,32
4,Afghanistan,2012,6
...,...,...,...
3804,Zimbabwe,2012,125
3805,Zimbabwe,2013,161
3806,Zimbabwe,2014,143
3807,Zimbabwe,2015,140


Count the countries that have export data from 2000-2016

In [8]:
import pandas as pd

start_year, end_year = 2000, 2016
years = pd.to_numeric(df_no_imports["year"], errors="coerce")
in_range = years.between(start_year, end_year)

years_per_country = (
    years[in_range]
    .groupby(df_no_imports.loc[in_range, "country_or_area"])
    .nunique()
)

required_years = end_year - start_year + 1
countries = years_per_country[years_per_country == required_years].index.sort_values()

print(f"Countries with data every year from {start_year} to {end_year}: {len(countries)}")
for country in countries:
    print(country)

Countries with data every year from 2000 to 2016: 93
Albania
Algeria
Argentina
Armenia
Aruba
Australia
Bahrain
Barbados
Belarus
Belgium
Belize
Benin
Botswana
Brazil
Bulgaria
Cambodia
Canada
Chile
China
China, Hong Kong SAR
Colombia
Costa Rica
Croatia
Cyprus
Czech Rep.
Denmark
Ecuador
Egypt
El Salvador
Estonia
Ethiopia
Finland
France
Georgia
Germany
Greece
Greenland
Guyana
Hungary
Iceland
India
Indonesia
Ireland
Israel
Italy
Jamaica
Japan
Jordan
Kazakhstan
Latvia
Lithuania
Luxembourg
Madagascar
Malaysia
Maldives
Malta
Mauritius
Mexico
Morocco
Namibia
Netherlands
New Zealand
Niger
Norway
Oman
Other Asia, nes
Panama
Paraguay
Peru
Philippines
Poland
Portugal
Qatar
Rep. of Korea
Rep. of Moldova
Romania
Russian Federation
Sao Tome and Principe
Senegal
Singapore
Slovakia
South Africa
Spain
Sri Lanka
Sweden
Switzerland
Tunisia
Turkey
USA
United Arab Emirates
United Kingdom
United Rep. of Tanzania
Uruguay


In [ ]:
import pandas as pd

years = pd.to_numeric(df_no_imports["year"], errors="coerce")
in_range = years.between(1990, 2016)

years_per_country = (
    years[in_range]
    .groupby(df_no_imports.loc[in_range, "country_or_area"])
    .nunique()
)

countries = years_per_country[years_per_country == 27].index.sort_values()

print(f"Countries with data every year from 1990 to 2016: {len(countries)}")
for country in countries:
    print(country)

We made a pivot table of all these countries and their non-null records

In [11]:
import pandas as pd

start_year, end_year = 2000, 2016
required_years = end_year - start_year + 1

# Select export records within the requested period
year = pd.to_numeric(df_no_imports["year"], errors="coerce")
flow = df_no_imports["flow"].astype("string").str.strip().str.lower()
keep = flow.eq("export") & year.between(start_year, end_year)

exports = df_no_imports.loc[keep].copy()
exports["year"] = year.loc[keep].astype(int)

# Count distinct export years for each country
years_per_country = exports.groupby("country_or_area")["year"].nunique()
eligible_countries = years_per_country[years_per_country == required_years].index

# Keep every export row for the qualifying countries
df_exports_2000_2016 = exports.loc[exports["country_or_area"].isin(eligible_countries)]

print(f"Countries with exports in all {required_years} years: {len(eligible_countries):,}")
print(f"Total export rows: {len(df_exports_2000_2016):,}")
display(df_exports_2000_2016)

Countries with exports in all 17 years: 93
Total export rows: 1,567,575


,country_or_area,year,comm_code,commodity,flow,trade_usd,weight_kg,quantity_name,quantity,category
6,Albania,2016,10511,"Fowls, live domestic < 185 grams",Export,87581,5320.0,Number of items,115180.0,01_live_animals
10,Albania,2016,10600,"Animals, live, except farm animals",Export,584025,59521.0,Number of items,59521.0,01_live_animals
18,Albania,2015,10600,"Animals, live, except farm animals",Export,636184,80774.0,Number of items,80694.0,01_live_animals
40,Albania,2013,10600,"Animals, live, except farm animals",Export,846588,99329.0,No Quantity,NaN,01_live_animals
43,Albania,2012,10290,"Bovine animals, live, except pure-bred breeding",Export,3390,2270.0,Number of items,22.0,01_live_animals
...,...,...,...,...,...,...,...,...,...,...
8225607,USA,2004,TOTAL,ALL COMMODITIES,Export,817905572144,NaN,No Quantity,NaN,all_commodities
8225610,USA,2003,TOTAL,ALL COMMODITIES,Export,723608647843,NaN,No Quantity,NaN,all_commodities
8225613,USA,2002,TOTAL,ALL COMMODITIES,Export,693222414198,NaN,No Quantity,NaN,all_commodities
8225616,USA,2001,TOTAL,ALL COMMODITIES,Export,731005997847,NaN,No Quantity,NaN,all_commodities


In [14]:
# Get the distinct commodity categories from the filtered export data
categories = (
    df_exports_2000_2016["category"]
    .dropna()
    .astype(str)
    .str.strip()
)
unique_categories = sorted(categories[categories != ""].unique())

# Print the count and all category names as one comma-separated string
categories_string = ", ".join(unique_categories)

print(f"Total distinct categories: {len(unique_categories)}")
print(categories_string)

Total distinct categories: 98
01_live_animals, 02_meat_and_edible_meat_offal, 03_fish_crustaceans_molluscs_aquatic_invertebrates_ne, 04_dairy_products_eggs_honey_edible_animal_product_nes, 05_products_of_animal_origin_nes, 06_live_trees_plants_bulbs_roots_cut_flowers_etc, 07_edible_vegetables_and_certain_roots_and_tubers, 08_edible_fruit_nuts_peel_of_citrus_fruit_melons, 09_coffee_tea_mate_and_spices, 10_cereals, 11_milling_products_malt_starches_inulin_wheat_glute, 12_oil_seed_oleagic_fruits_grain_seed_fruit_etc_ne, 13_lac_gums_resins_vegetable_saps_and_extracts_nes, 14_vegetable_plaiting_materials_vegetable_products_nes, 15_animal_vegetable_fats_and_oils_cleavage_products_et, 16_meat_fish_and_seafood_food_preparations_nes, 17_sugars_and_sugar_confectionery, 18_cocoa_and_cocoa_preparations, 19_cereal_flour_starch_milk_preparations_and_products, 20_vegetable_fruit_nut_etc_food_preparations, 21_miscellaneous_edible_preparations, 22_beverages_spirits_and_vinegar, 23_residues_wastes_of_fo

In [15]:
import pandas as pd

categories = df_exports_2000_2016["category"].dropna().str.strip()
category_table = pd.DataFrame({
    "Category": sorted(categories[categories != ""].unique())
})
category_table.index = pd.RangeIndex(1, len(category_table) + 1, name="No.")

print(f"Total distinct categories: {len(category_table)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(category_table)

Total distinct categories: 98


,Category
No.,
1,01_live_animals
2,02_meat_and_edible_meat_offal
3,03_fish_crustaceans_molluscs_aquatic_invertebrates_ne
4,04_dairy_products_eggs_honey_edible_animal_product_nes
5,05_products_of_animal_origin_nes
6,06_live_trees_plants_bulbs_roots_cut_flowers_etc
7,07_edible_vegetables_and_certain_roots_and_tubers
8,08_edible_fruit_nuts_peel_of_citrus_fruit_melons
9,09_coffee_tea_mate_and_spices


In [16]:
import pandas as pd

commodity_table = (
    df_exports_2000_2016
    .groupby(["category", "country_or_area"])["comm_code"]
    .nunique()
    .unstack("country_or_area", fill_value=0)
    .rename_axis(index="Commodity category", columns="Country")
    .sort_index()
    .sort_index(axis=1)
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(commodity_table)

Country,Albania,Algeria,Argentina,Armenia,Aruba,Australia,Bahrain,Barbados,Belarus,Belgium,Belize,Benin,Botswana,Brazil,Bulgaria,Cambodia,Canada,Chile,China,"China, Hong Kong SAR",Colombia,Costa Rica,Croatia,Cyprus,Czech Rep.,Denmark,Ecuador,Egypt,El Salvador,Estonia,Ethiopia,Finland,France,Georgia,Germany,Greece,Greenland,Guyana,Hungary,Iceland,India,Indonesia,Ireland,Israel,Italy,Jamaica,Japan,Jordan,Kazakhstan,Latvia,Lithuania,Luxembourg,Madagascar,Malaysia,Maldives,Malta,Mauritius,Mexico,Morocco,Namibia,Netherlands,New Zealand,Niger,Norway,Oman,"Other Asia, nes",Panama,Paraguay,Peru,Philippines,Poland,Portugal,Qatar,Rep. of Korea,Rep. of Moldova,Romania,Russian Federation,Sao Tome and Principe,Senegal,Singapore,Slovakia,South Africa,Spain,Sri Lanka,Sweden,Switzerland,Tunisia,Turkey,USA,United Arab Emirates,United Kingdom,United Rep. of Tanzania,Uruguay
Commodity category,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
01_live_animals,8,6,11,12,10,15,11,13,14,15,7,7,14,15,15,8,15,14,15,7,13,11,15,14,15,15,7,15,13,13,11,11,15,10,15,15,1,12,15,4,15,14,15,12,15,11,9,11,14,14,14,14,6,15,1,11,8,14,8,15,15,15,15,13,11,12,8,6,13,7,15,15,11,10,9,15,15,6,15,13,15,15,15,13,15,15,9,12,15,14,15,15,11
02_meat_and_edible_meat_offal,24,21,43,23,27,49,48,20,39,49,11,16,48,48,48,11,49,45,47,47,36,36,46,43,49,49,24,44,41,49,36,48,49,35,49,49,15,38,48,44,47,47,49,27,49,27,32,33,40,48,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
03_fish_crustaceans_molluscs_aquatic_invertebrates_ne,65,59,56,28,35,87,60,42,64,87,37,29,51,75,84,51,87,77,85,81,56,66,80,59,87,87,68,84,54,83,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
04_dairy_products_eggs_honey_edible_animal_product_nes,20,19,24,18,19,25,23,20,25,25,13,13,25,25,25,10,25,23,24,25,25,24,25,23,25,25,20,25,24,25,22,25,25,21,25,25,2,24,25,24,25,25,25,22,25,22,25,23,23,25,25,25,20,25,4,21,22,24,22,25,25,25,21,24,22,24,20,14,25,23,25,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
05_products_of_animal_origin_nes,7,2,14,6,4,17,8,9,15,17,5,5,15,17,15,9,17,15,17,17,16,11,14,12,16,17,14,16,9,14,16,15,17,9,17,14,6,9,17,12,17,17,16,9,17,6,17,7,8,14,15,14,11,15,2,7,10,16,11,17,17,17,6,17,9,15,13,10,13,11,17,16,7,16,8,16,16,1,10,16,15,17,17,14,17,17,11,16,17,16,17,17,13
06_live_trees_plants_bulbs_roots_cut_flowers_etc,10,7,11,9,6,12,11,10,11,12,7,4,10,12,12,7,12,11,12,12,12,12,12,10,12,12,11,13,11,12,11,12,12,12,12,12,3,9,12,12,12,12,12,11,12,10,12,12,11,12,12,12,10,12,0,8,10,12,12,12,12,12,7,12,10,12,9,7,10,10,12,12,11,12,9,12,12,8,10,12,12,12,12,12,12,12,11,12,12,12,12,12,8
07_edible_vegetables_and_certain_roots_and_tubers,51,51,55,41,23,56,54,46,50,56,24,26,56,56,54,32,56,55,56,54,56,55,55,54,55,55,53,57,50,55,54,53,56,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
08_edible_fruit_nuts_peel_of_citrus_fruit_melons,39,35,48,38,23,50,48,43,48,50,25,23,50,49,50,26,50,49,50,49,46,48,49,46,49,50,42,50,41,49,37,50,50,47,50,50,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
09_coffee_tea_mate_and_spices,24,25,31,27,13,32,31,31,30,32,18,11,31,31,32,19,32,31,31,32,32,31,32,29,32,32,29,33,27,32,32,32,32,30,32,32,4,32,32,28,32,32,31,30,32,31,30,29,27,32,32,32,32,32,3,23,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
category_ranking = (
    df_exports_2000_2016
    .groupby("category")["comm_code"]
    .nunique()
    .reset_index(name="distinct_commodities")
    .sort_values(["distinct_commodities", "category"], ignore_index=True)
)

with pd.option_context("display.max_rows", None):
    display(category_ranking)

,category,distinct_commodities
0,all_commodities,1
1,99_commodities_not_specified_according_to_kind,2
2,46_manufactures_of_plaiting_material_basketwor...,6
3,45_cork_and_articles_of_cork,7
4,66_umbrellas_walking_sticks_seat_sticks_whips_etc,7
5,36_explosives_pyrotechnics_matches_pyrophorics...,8
6,67_bird_skin_feathers_artificial_flowers_human...,8
7,80_tin_and_articles_thereof,8
8,24_tobacco_and_manufactured_tobacco_substitutes,9
9,50_silk,10


In [19]:
# Get the distinct commodity categories from the filtered export data
categories = (
    df_exports_2000_2016["category"]
    .dropna()
    .astype(str)
    .str.strip()
)
unique_categories = sorted(categories[categories != ""].unique())

# Print the count and all category names as one comma-separated string
categories_string = ", ".join(unique_categories)

print(f"Total distinct categories: {len(unique_categories)}")
print(categories_string)

Total distinct categories: 98
01_live_animals, 02_meat_and_edible_meat_offal, 03_fish_crustaceans_molluscs_aquatic_invertebrates_ne, 04_dairy_products_eggs_honey_edible_animal_product_nes, 05_products_of_animal_origin_nes, 06_live_trees_plants_bulbs_roots_cut_flowers_etc, 07_edible_vegetables_and_certain_roots_and_tubers, 08_edible_fruit_nuts_peel_of_citrus_fruit_melons, 09_coffee_tea_mate_and_spices, 10_cereals, 11_milling_products_malt_starches_inulin_wheat_glute, 12_oil_seed_oleagic_fruits_grain_seed_fruit_etc_ne, 13_lac_gums_resins_vegetable_saps_and_extracts_nes, 14_vegetable_plaiting_materials_vegetable_products_nes, 15_animal_vegetable_fats_and_oils_cleavage_products_et, 16_meat_fish_and_seafood_food_preparations_nes, 17_sugars_and_sugar_confectionery, 18_cocoa_and_cocoa_preparations, 19_cereal_flour_starch_milk_preparations_and_products, 20_vegetable_fruit_nut_etc_food_preparations, 21_miscellaneous_edible_preparations, 22_beverages_spirits_and_vinegar, 23_residues_wastes_of_fo

In [18]:
import pandas as pd

data = df_exports_2000_2016
keys = ["country_or_area", "comm_code"]

# One row per country, commodity, and year with positive recorded export value
history = (
    data.loc[
        pd.to_numeric(data["trade_usd"], errors="coerce").gt(0),
        keys + ["year"]
    ]
    .dropna()
    .drop_duplicates()
    .sort_values(keys + ["year"])
    .copy()
)

# With unique, sorted annual records, a three-year span across four rows
# means all four years are consecutive.
year_three_records_back = history.groupby(keys)["year"].shift(3)
history["four_consecutive_years"] = history["year"].sub(year_three_records_back).eq(3)

coverage = (
    history.groupby("comm_code")
    .agg(
        countries_with_exports=("country_or_area", "nunique"),
        observations_with_growth_history=("four_consecutive_years", "sum")
    )
    .sort_values("observations_with_growth_history")
    .reset_index()
)

display(coverage)

,comm_code,countries_with_exports,observations_with_growth_history
0,847040,1,0
1,720832,1,0
2,20750,1,0
3,844220,3,0
4,720934,1,0
...,...,...,...
4991,190530,92,1153
4992,170490,92,1162
4993,210690,93,1182
4994,190590,91,1185


In [20]:
import pandas as pd

# Each number is the prefix of a category, such as "09_coffee_tea_mate_and_spices".
commodity_groups = {
    "animals_meat_seafood_dairy": [1, 2, 3, 4, 5, 16],
    "plants_and_crops": [6, 7, 8, 9, 10, 12, 13, 14],
    "processed_foods_oils_beverages_feed": [11, 15, 17, 18, 19, 20, 21, 22, 23],
    "tobacco": [24],
    "minerals_and_fuels": [25, 26, 27],
    "chemicals_and_related_products": list(range(28, 39)),
    "plastics_and_rubber": [39, 40],
    "hides_leather_and_fur": [41, 42, 43],
    "wood_paper_and_printed_goods": list(range(44, 50)),
    "textiles_and_clothing": list(range(50, 64)),
    "footwear_and_accessories": [64, 65, 66, 67],
    "stone_ceramics_and_glass": [68, 69, 70],
    "precious_stones_and_metals": [71],
    "base_metals_and_metal_goods": [72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83],
    "machinery_and_electronics": [84, 85],
    "transportation_equipment": [86, 87, 88, 89],
    "precision_instruments_and_timepieces": [90, 91],
    "musical_instruments": [92],
    "arms_and_ammunition": [93],
    "furniture_lighting_and_buildings": [94],
    "recreation_art_and_collectibles": [95, 97],
    "other_manufactured_goods": [96],
    "unspecified_commodities": [99],
}

# Keep only rows labeled Export and remove the all_commodities total.
source = df_exports_2000_2016
flow = source["flow"].astype("string").str.strip().str.lower()
category = source["category"].astype("string").str.strip()
exports = source.loc[flow.eq("export") & category.ne("all_commodities")]

# Match each two-digit category prefix to its product family.
group_by_code = {
    f"{code:02d}": group
    for group, codes in commodity_groups.items()
    for code in codes
}
category_codes = exports["category"].str.strip().str.extract(r"^(\d{2})_", expand=False)
product_family = category_codes.map(group_by_code)

# Store the 23 DataFrames by name. Each keeps the original columns.
grouped = exports.groupby(product_family)
export_datasets = {
    name: grouped.get_group(name) if name in grouped.groups else exports.iloc[0:0].copy()
    for name in commodity_groups
}

# Display the number of export records in each dataset.
summary = pd.DataFrame(
    [(name, len(data)) for name, data in export_datasets.items()],
    columns=["dataset", "export_rows"]
)

print(f"Datasets created: {len(export_datasets)}")
display(summary)

Datasets created: 23


,dataset,export_rows
0,animals_meat_seafood_dairy,104834
1,plants_and_crops,121730
2,processed_foods_oils_beverages_feed,160908
3,tobacco,9163
4,minerals_and_fuels,57287
5,chemicals_and_related_products,184246
6,plastics_and_rubber,36376
7,hides_leather_and_fur,51453
8,wood_paper_and_printed_goods,80519
9,textiles_and_clothing,249089


In [24]:
df_animals_meat_seafood_dairy = export_datasets["animals_meat_seafood_dairy"]
display(df_animals_meat_seafood_dairy)

,country_or_area,year,comm_code,commodity,flow,trade_usd,weight_kg,quantity_name,quantity,category
6,Albania,2016,10511,"Fowls, live domestic < 185 grams",Export,87581,5320.0,Number of items,115180.0,01_live_animals
10,Albania,2016,10600,"Animals, live, except farm animals",Export,584025,59521.0,Number of items,59521.0,01_live_animals
18,Albania,2015,10600,"Animals, live, except farm animals",Export,636184,80774.0,Number of items,80694.0,01_live_animals
40,Albania,2013,10600,"Animals, live, except farm animals",Export,846588,99329.0,No Quantity,NaN,01_live_animals
43,Albania,2012,10290,"Bovine animals, live, except pure-bred breeding",Export,3390,2270.0,Number of items,22.0,01_live_animals
...,...,...,...,...,...,...,...,...,...,...
1334900,Russian Federation,2000,160430,Caviar and caviar substitutes prepared from fi...,Export,21590888,471756.0,Weight in kilograms,471756.0,16_meat_fish_and_seafood_food_preparations_nes
1334902,Russian Federation,2000,160510,"Crab, prepared or preserved",Export,13528843,1440861.0,Weight in kilograms,1440861.0,16_meat_fish_and_seafood_food_preparations_nes
1334904,Russian Federation,2000,160520,"Shrimps and prawns, prepared or preserved",Export,170960,103425.0,Weight in kilograms,103425.0,16_meat_fish_and_seafood_food_preparations_nes
1334907,Russian Federation,2000,160540,"Crustaceans nes, prepared or preserved",Export,10780,21805.0,Weight in kilograms,21805.0,16_meat_fish_and_seafood_food_preparations_nes
